---
# **Lab: PyTorch autograd**
---

# ▶️ CUDA tools...

In [ ]:
!nvidia-smi

**Numba code used as a CUDA sanity check**:
-   Imports Numba, a JIT compiler that accelerates Python code.
-	numba.cuda provides GPU (CUDA) support using NVIDIA GPUs.
	- ✔ Confirms NumPy and Numba are installed
	- ✔ Confirms CUDA drivers are visible
	- ✔ Confirms GPU compute capability
	- ✔ Helps debug environment issues before running GPU kernels

Probes the system for available CUDA-capable GPUs. Prints:
-	Number of GPUs
-	GPU names
-	Compute capability
-	Driver/runtime status


In [ ]:
import numpy as np
import numba
from numba import cuda
import warnings
warnings.filterwarnings("ignore")

print(np.__version__)
print(numba.__version__)

cuda.detect()



# ✅ Gradient computation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Function and gradient
def f(x, y):
    return x**2 + y**2

def grad_f(x, y):
    return np.array([2*x, 2*y])

# Grid for surface and contour
x = np.linspace(-3, 3, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

# Point of interest
x0, y0 = 2.5, -2.1
gx, gy = grad_f(x0, y0)
z0 = f(x0, y0)
print(gx,gy)

# -------------------------------
# 1. 3D Surface Plot
# -------------------------------
fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)

# plot the point
ax1.scatter(x0, y0, z0, color='red', s=50)

# draw gradient direction in 3D (extruded at z0 level)
ax1.quiver(x0, y0, z0,
           gx, gy, 0,
           length=0.5, color='red')

ax1.set_title(r"3D Surface: $f(x,y)=x^2+y^2$, gradient at (2.5,-2)")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("z")

# -------------------------------
# 2. Contour + Gradient Vector
# -------------------------------
# Point of interest
x0, y0 = .7, -.7
gx, gy = grad_f(x0, y0)
z0 = f(x0, y0)

ax2 = fig.add_subplot(122)
contours = ax2.contour(X, Y, Z, levels=20, cmap='viridis')
ax2.clabel(contours)

# point
ax2.plot(x0, y0, 'ro', markersize=6)

# gradient vector
ax2.quiver(x0, y0, gx, gy,
           angles='xy', scale_units='xy', scale=1,
           color='red', width=0.005)

ax2.set_aspect('equal')
ax2.set_title("Contour Plot with Gradient at (0.7,-0.7)")
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.grid(True)

plt.show()

# ✅ A Simple Example

Let’s start with a straightforward example. First, we’ll do some imports to let us graph our results:

In [ ]:
# First Example with Autograd
import torch

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

z = x**2 + y**2
print("z =", z)

z.backward()
print("dz/dx =", x.grad) # dz/dx = 2*x
print("dz/dy =", y.grad) # dz/dy = 2*y


In [ ]:
a = torch.tensor(4.0, requires_grad=True)
b = torch.tensor(2.0, requires_grad=True)

f = (a * b) + (a ** 2)
f.backward()

print("df/da =", a.grad)  # b + 2a
print("df/db =", b.grad)  # a

Let's compute the derivative of $sin(x)$ at $x$ in $[0,2\pi]$ using PyTorch's automatic differentiation
([link](https://pytorch.org/docs/stable/autograd.html)) and specify `requires_grad=True`(like most functions that create tensors, `torch.linspace()` accepts an optional `requires_grad` option). 

Setting this flag means that in every computation that follows, autograd will be accumulating the history of the computation in the output tensors of that computation.


In [ ]:
import torch
import numpy as np
import plotly.graph_objs as go
from torch.autograd import grad

x = torch.linspace(0, 2*np.pi, 100, requires_grad=True)  # enables automatic different
y = torch.sin(x)
dy = grad(y.sum(), x)[0]  # here's the magic function `grad`

# plotting with plotly
trace_sin = go.Scatter(x=x.detach().numpy(), y=y.detach().numpy(), mode='lines', name='y=Sine')
trace_deriv = go.Scatter(x=x.detach().numpy(), y=dy.detach().numpy(), mode='lines', name='Derivative of Sine')
fig_plotly = go.Figure([trace_sin, trace_deriv])
fig_plotly.update_layout(xaxis_title='$x$', yaxis_title='y', title='Sine and its Derivative')
fig_plotly.show()

print('x,', x)	
print('y,', y)
print('dy,', dy)


# ✅ Complete Pytorch NN training example



This script demonstrates a complete neural network training workflow using PyTorch,
including:
- Data preparation
- Model definition
- Loss function and optimizer setup
- Complete training loop with autograd
- Evaluation and visualization


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ==============================================================================
# STEP 1: CREATE SYNTHETIC CLASSIFICATION DATASET
# ==============================================================================

def create_synthetic_data(num_samples=2000, num_classes=4):
    """
    Create a 2D synthetic dataset with specified number of classes.
    Each class forms a cluster in 2D space.
    """
    # Cluster centers
    centers = torch.tensor([
        [2.0, 2.0],
        [-2.0, 2.0],
        [-2.0, -2.0],
        [2.0, -2.0]
    ], dtype=torch.float32)
    
    # Generate samples around each center
    X_list = []
    y_list = []
    samples_per_class = num_samples // num_classes
    
    for i in range(num_classes):
        cluster = torch.randn(samples_per_class, 2) * 0.8 + centers[i]
        X_list.append(cluster)
        y_list.append(torch.full((samples_per_class,), i, dtype=torch.long))
    
    X = torch.cat(X_list, dim=0)
    y = torch.cat(y_list, dim=0)
    
    # Shuffle data
    perm = torch.randperm(num_samples)
    X = X[perm]
    y = y[perm]
    
    return X, y

# Create dataset
num_samples = 2000
num_classes = 4
feature_dim = 2

X, y = create_synthetic_data(num_samples, num_classes)

# Train/test split (80/20)
n_train = int(0.8 * num_samples)
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print(f"Total samples: {num_samples}")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Feature dimension: {feature_dim}")
print(f"Number of classes: {num_classes}")



STEP 2: DEFINE NEURAL NETWORK MODEL... 

In [ ]:
class NeuralNetwork(nn.Module):
    """
    A feedforward neural network for multi-class classification.
    
    Architecture:
        Input (2) -> FC1 (64) -> ReLU -> Dropout -> 
        FC2 (32) -> ReLU -> Dropout -> 
        FC3 (4) -> Output
    
    Args:
        input_dim: Number of input features
        hidden_dim1: Size of first hidden layer
        hidden_dim2: Size of second hidden layer
        output_dim: Number of output classes
    """
    
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, output_dim):
        super(NeuralNetwork, self).__init__()
        
        # Layer 1: Input -> Hidden1
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        
        # Layer 2: Hidden1 -> Hidden2
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        
        # Layer 3: Hidden2 -> Output
        self.fc3 = nn.Linear(hidden_dim2, output_dim)
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, input_dim)
            
        Returns:
            Output tensor of shape (batch_size, output_dim)
        """
        # Layer 1: Linear + ReLU + Dropout
        x = self.fc1(x)
        x = F.relu(x)
        
        # Layer 2: Linear + ReLU + Dropout
        x = self.fc2(x)
        x = F.relu(x)
        
        # Output layer (raw logits, no activation)
        # CrossEntropyLoss will apply softmax internally
        x = self.fc3(x)
        
        return x

# Create model instance
input_dim = 2
hidden_dim1 = 64
hidden_dim2 = 32
output_dim = num_classes

model = NeuralNetwork(input_dim, hidden_dim1, hidden_dim2, output_dim)

print("\n" + "=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")



## ↘️ TODO...

STEP 3: define loss function and optimizer for training loop...

In [ ]:
# Loss function: Cross Entropy Loss

# TODO

# ==============================================================================
# STEP 4: TRAINING LOOP
# ==============================================================================

def train_model(model, X_train, y_train, criterion, optimizer, num_epochs=50, batch_size=32):
    """
    Complete training loop with gradient computation.
    
    The training process follows these steps for each batch:
    1. Zero gradients (optimizer.zero_grad())
    2. Forward pass (model(inputs))
    3. Compute loss (criterion(outputs, labels))
    4. Backward pass (loss.backward()) - AUTOGRAD MAGIC!
    5. Update parameters (optimizer.step())
    """
    
    # Lists to store metrics
    train_losses = []
    train_accuracies = []
    
    num_batches = len(X_train) // batch_size
    
    print("\n" + "=" * 60)
    print("STARTING TRAINING")
    print("=" * 60)
    
    for epoch in range(num_epochs):
        # Set model to training mode
        # This enables dropout and batch normalization updates
        model.train()
        
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        # Mini-batch training
        for i in range(num_batches):
            # Get batch
            start_idx = i * batch_size
            end_idx = start_idx + batch_size
            
            inputs = X_train[start_idx:end_idx]
            labels = y_train[start_idx:end_idx]
            
            # ---------------------------------------------------------
            # STEP 4.1: ZERO GRADIENTS
            # Clear accumulated gradients from previous iteration
            # CRITICAL: PyTorch accumulates gradients by default!
            # ---------------------------------------------------------
            
            # TODO
            
            # ---------------------------------------------------------
            # STEP 4.2: FORWARD PASS
            # Compute predictions by passing inputs through the model
            # This builds the computational graph for autograd
            # ---------------------------------------------------------
            
            # TODO
            
            # ---------------------------------------------------------
            # STEP 4.3: COMPUTE LOSS
            # Measure how far predictions are from true labels
            # ---------------------------------------------------------
            
            # TODO
                        
            # ---------------------------------------------------------
            # STEP 4.4: BACKWARD PASS (AUTOMATIC DIFFERENTIATION)
            # Compute gradients of loss w.r.t. all model parameters
            # This is where PyTorch's autograd engine shines!
            # It traverses the computational graph in reverse,
            # applying the chain rule to compute all gradients
            # ---------------------------------------------------------
            
            # TODO
            
            # ---------------------------------------------------------
            # STEP 4.5: UPDATE PARAMETERS
            # Optimizer uses computed gradients to update weights
            # Uses Adam's adaptive learning rate mechanism
            # ---------------------------------------------------------
            
            # TODO
            
            # Accumulate statistics
            
            # TODO
                        
            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        # Calculate epoch metrics
        avg_loss = epoch_loss / num_batches
        train_accuracy = 100 * correct / total
        
        train_losses.append(avg_loss)
        train_accuracies.append(train_accuracy)
        
        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Accuracy: {train_accuracy:.2f}% | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    print("=" * 60)
    print("TRAINING COMPLETE")
    print("=" * 60)
    
    return train_losses, train_accuracies

# Train the model
train_losses, train_accuracies = train_model(
    model, X_train, y_train, criterion, optimizer, num_epochs=50, batch_size=32
)

# ==============================================================================
# STEP 5: EVALUATION
# ==============================================================================

def evaluate_model(model, X, y):
    """
    Evaluate model performance on given data.
    Uses torch.no_grad() to disable gradient computation for efficiency.
    """
    model.eval()  # Set to evaluation mode
    
    # Disable gradient computation
    # This saves memory and speeds up inference
    with torch.no_grad():
        outputs = model(X)
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).float().mean().item() * 100
    
    return accuracy, predicted

# Evaluate on train and test sets
train_acc, train_pred = evaluate_model(model, X_train, y_train)
test_acc, test_pred = evaluate_model(model, X_test, y_test)

print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)
print(f"Training Accuracy: {train_acc:.2f}%")
print(f"Test Accuracy: {test_acc:.2f}%")

# Confusion Matrix
print("\nConfusion Matrix (Test Set):")
cm = confusion_matrix(y_test.numpy(), test_pred.numpy())
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test.numpy(), test_pred.numpy(), 
                           target_names=[f'Class {i}' for i in range(num_classes)]))



STEP 6: visualization

In [ ]:
def plot_training_progress(train_losses, train_accuracies):
    """Plot training loss and accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curve
    axes[0].plot(train_losses, 'b-', linewidth=2, label='Training Loss')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training Loss Over Epochs', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # Accuracy curve
    axes[1].plot(train_accuracies, 'b-', linewidth=2, label='Training Accuracy')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Training Accuracy Over Epochs', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    axes[1].set_ylim([90, 100])
    
    plt.tight_layout()
    plt.savefig('training_progress.png', dpi=150, bbox_inches='tight')
    plt.show()

def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    """Plot the decision boundary of the trained model."""
    model.eval()
    
    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Convert to tensor and predict
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    
    with torch.no_grad():
        Z = model(grid)
        _, Z = torch.max(Z, 1)
    
    Z = Z.numpy().reshape(xx.shape)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Contour plot
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='viridis')
    
    # Scatter plot of data points
    colors = ['red', 'blue', 'green', 'orange']
    for i in range(num_classes):
        mask = y == i
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[i], 
                  label=f'Class {i}', edgecolors='k', s=50)
    
    ax.set_xlabel('Feature 1', fontsize=12)
    ax.set_ylabel('Feature 2', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('decision_boundary.png', dpi=150, bbox_inches='tight')
    plt.show()

# Generate visualizations
print("\nGenerating visualizations...")
plot_training_progress(train_losses, train_accuracies)
plot_decision_boundary(model, X_test, y_test, "Decision Boundary (Test Data)")

In [ ]:
# ==============================================================================
# STEP 7: GRADIENT INSPECTION (AUTOGRAD DEMONSTRATION)
# ==============================================================================

print("\n" + "=" * 60)
print("GRADIENT INSPECTION (AUTOGRAD DEMONSTRATION)")
print("=" * 60)

# Take a single sample for demonstration
sample_input = X_train[0:1]
sample_label = y_train[0:1]

print(f"\nSample input: {sample_input}")
print(f"True label: {sample_label.item()}")

# Forward pass
model.train()
optimizer.zero_grad()

output = model(sample_input)
print(f"\nModel output (logits): {output.detach().numpy()}")

# Compute loss
loss = criterion(output, sample_label)
print(f"Loss: {loss.item():.4f}")

# Backward pass - AUTOGRAD COMPUTES GRADIENTS
loss.backward()

print("\n" + "-" * 60)
print("GRADIENTS COMPUTED BY AUTOGRAD:")
print("-" * 60)

# Inspect gradients for each parameter
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        print(f"{name:20s} | Shape: {str(param.shape):15s} | Grad norm: {grad_norm:.6f}")
    else:
        print(f"{name:20s} | NO GRADIENT")

print("-" * 60)

# ==============================================================================
# SUMMARY
# ==============================================================================

print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Model: NeuralNetwork ({input_dim}-{hidden_dim1}-{hidden_dim2}-{output_dim})")
print(f"Total Parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Training Epochs: 50")
print(f"Batch Size: 32")
print(f"Optimizer: Adam (lr=0.001)")
print(f"Final Training Accuracy: {train_acc:.2f}%")
print(f"Final Test Accuracy: {test_acc:.2f}%")
print("=" * 60)

print("\nKey Takeaways:")
print("1. optimizer.zero_grad() clears gradients from previous iterations")
print("2. loss.backward() computes gradients using automatic differentiation")
print("3. optimizer.step() updates parameters using computed gradients")
print("4. torch.no_grad() disables gradient computation during evaluation")
print("5. model.train() and model.eval() control dropout and batch norm behavior")

# ✅ Regression example with autograd

In [ ]:
import torch
import plotly.graph_objs as go

def F2C(x):
    return (x - 32) * 5.0/9.0
def C2F(x):
    return x * 9.0/5.0 + 32

def termometer(N, noise=5):
    c = torch.rand(N)*100   # Celsius from 30 to 42 
    f = C2F(c)
    return c, f + noise*torch.randn(N)

## Generate data
t_c, t_u = termometer(20, noise=1)

trace = go.Scatter(x=t_u.detach().numpy(), 
                   y=t_c.detach().numpy(), 
                   mode='markers', 
                   marker=dict(color='red', size=6), 
                   name='t_u vs t_c')
fig = go.Figure(trace)
fig.update_layout(title='Temperature: U vs C',
                  xaxis_title='Temperature in U',
                  yaxis_title='Temperature in C')
fig.show()


In [ ]:
import torch
import numpy as np

def model(t, w, b):
    return w * t + b

def loss_fn(t_p, t_c):
    diffs = (t_p - t_c)**2
    return diffs.mean()

w = torch.tensor(1.0)
b = torch.tensor(0.0)

print('Initial parameters:', w, b)

In [ ]:
# model prediction pre-training
t_p = model(t_u, w, b)
t_p

In [ ]:
# loss pre-training
loss = loss_fn(t_p, t_c)
loss

In [ ]:
def dloss_fn(t_p, t_c):
    """ Computes the derivative of the loss function with respect to t_p.
    
    Params:
        t_p: Predicted temperatures
        t_c: Actual temperatures
        
    Returns:
        dsq_diffs: Derivative of the loss with respect to t_p
    """
    dsq_diffs = 2 * (t_p - t_c) / t_p.size(0)
    return dsq_diffs

def dmodel_dw(t_u, w, b):
    """ Computes the derivative of the model with respect to w.
    Params:
        t_u: Input temperatures
        w: Weight parameter
        b: Bias parameter
    Returns:
        t_u: Derivative of the model with respect to w
    """
    return t_u

def dmodel_db(t_u, w, b):
    """ Computes the derivative of the model with respect to b.     

    Params:
        t_u: Input temperatures
        w: Weight parameter
        b: Bias parameter
    Returns:
        1.0: Derivative of the model with respect to b  
    """
    return 1.0

def grad_fn(t_u, t_c, t_p, w, b):
    dloss_dtp = dloss_fn(t_p, t_c)
    dloss_dw = dloss_dtp * dmodel_dw(t_u, w, b)
    dloss_db = dloss_dtp * dmodel_db(t_u, w, b)
    return torch.stack([dloss_dw.sum(), dloss_db.sum()])

In [ ]:
def training_loop(n_epochs, learning_rate, params, t_u, t_c):
    for epoch in range(1, n_epochs + 1):

        # parameters
        w, b = params

        # forward pass and loss computation
        t_p = model(t_u, w, b)
        loss = loss_fn(t_p, t_c)
        grad = grad_fn(t_u, t_c, t_p, w, b)
        
        # update parameters
        params -= learning_rate * grad

        if epoch % 100 == 0:
            print('Epoch %d, Loss %f' % (epoch, float(loss)))
            

    return params, grad


t_un = 0.1 * t_u

params, grad = training_loop(
    n_epochs=5000,
    learning_rate=0.005,
    params = torch.tensor([1.0, 0.0]),
    t_u = t_un,
    t_c = t_c
)
print('\nTrained parameters:')
print('    w = %f, b = %f' % (params[0], params[1]))
print('    grad_w = %f, grad_b = %f' % (grad[0], grad[1]))

In [ ]:
# loss post-training
w, b = params[0], params[1]   
t_p = model(t_un, w, b) 

loss = loss_fn(t_p, t_c)
loss

In [ ]:
# model prediction post-training
x = torch.linspace(0, t_u.max(), 100)
y = model(x*0.1, w, b)
trace1 = go.Scatter(x=x.detach().numpy(), 
                   y=y.detach().numpy(), 
                   mode='lines', 
                   line=dict(color='gray'), 
                   name='linear fit')

fig = go.Figure([trace, trace1])
fig.update_layout(title='Temperature: U vs C (After Training)',
                  xaxis_title='Temperature in U',
                  yaxis_title='Temperature in C')
fig.show()



Using torch.grad to compute gradients for a simple linear regression model.

```python
torch.grad

In [ ]:
def training_loop(n_epochs, learning_rate, params, t_u, t_c):
    for epoch in range(1, n_epochs + 1):

        # 1) clear old gradients
        if params.grad is not None:
            params.grad.zero_()

        # 2) forward + loss
        t_p = model(t_u, params[0], params[1])
        loss = loss_fn(t_p, t_c)

        # 3) backward
        loss.backward()

        # 4) update (do not track this in autograd)
        with torch.no_grad():
            params -= learning_rate * params.grad

        if epoch % 500 == 0:
            print(f"Epoch {epoch}, Loss {float(loss):.6f}")

    return params


params = training_loop(
    n_epochs=5000,
    learning_rate=0.005,
    params=torch.tensor([1.0, 0.0], requires_grad=True),
    t_u=t_un,
    t_c=t_c
)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# Simple linear model: F = w*C + b
model = nn.Linear(1, 1)

# Loss function and optimizer
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.005)


# Training loop
epochs = 5000
for epoch in range(epochs):
    
    # Forward pass
    pred = model(t_un.unsqueeze(1))
    loss = loss_fn(t_c.unsqueeze(1), pred)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print progress every 500 epochs
    if epoch % 500 == 0:
        w, b = model.parameters()
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}, w = {w.item():.4f}, b = {b.item():.4f}")

# Final parameters
w, b = model.parameters()
print(f"\nLearned parameters: w = {w.item():.4f}, b = {b.item():.4f}")

# Test prediction
test_celsius = torch.tensor([100.0]).unsqueeze(0)
pred_fahrenheit = model(test_celsius)
print(f"Prediction for 100°C: {pred_fahrenheit.item():.2f}°F")

